# Bronze Layer Data Validation
This notebook runs data integrity tests on the `bronze` tables to ensure the Medallion Pipeline correctly ingested the raw data into DuckDB.
> Note: The raw data and `.db` file are excluded from source control.

In [1]:
import duckdb
import pandas as pd

# Connect to the Lakehouse DB (read-only to avoid lock conflicts)
conn = duckdb.connect('../lakehouse_db/aki_lakehouse.db', read_only=True)
print('Connected to Lakehouse Database successfully.')

Connected to Lakehouse Database successfully.


### Test 1: Materialization Verification
Ensure that all bronze tables have been successfully populated from the raw `.ndjson` files.

In [2]:
tables = [
    'bronze_patients', 'bronze_encounters', 'bronze_conditions', 
    'bronze_chartevents', 'bronze_labevents', 'bronze_outputevents', 'bronze_medications'
]

for table in tables:
    count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table}: {count:,} rows')

bronze_patients: 299,712 rows
bronze_encounters: 73,181 rows
bronze_conditions: 4,756,326 rows
bronze_chartevents: 313,645,032 rows
bronze_labevents: 118,171,367 rows
bronze_outputevents: 4,234,967 rows
bronze_medications: 8,978,893 rows


### Test 2: Nested JSON Structure Validation
Verify that DuckDB natively parsed the deeply nested FHIR data structures into native structs/arrays, rather than treating them as flat strings.

In [3]:
nested_sample = conn.execute('''
    SELECT 
        id,
        subject,
        code,
        valueQuantity
    FROM bronze_labevents
    LIMIT 10
''').df()
display(nested_sample)

,id,subject,code,valueQuantity
0,f63ed104-2867-58f6-a875-090f65bbf302,{'reference': 'Patient/00000027-c5e0-554f-8e85...,"{'coding': [{'code': '50804', 'system': 'http:...","{'code': 'mEq/L', 'unit': 'mEq/L', 'value': 25..."
1,6fceb05c-4555-5964-ad81-06875cdad7bc,{'reference': 'Patient/00000027-c5e0-554f-8e85...,"{'coding': [{'code': '51492', 'system': 'http:...",<NA>
2,f3cb3014-9846-5411-8c42-992f97d1bcfe,{'reference': 'Patient/00000027-c5e0-554f-8e85...,"{'coding': [{'code': '50828', 'system': 'http:...",<NA>
3,ced7657a-8e8b-5798-96ae-abc24ff379ca,{'reference': 'Patient/00000027-c5e0-554f-8e85...,"{'coding': [{'code': '52033', 'system': 'http:...",<NA>
4,d95efae4-c0ec-532b-8069-3c509969e011,{'reference': 'Patient/00000027-c5e0-554f-8e85...,"{'coding': [{'code': '50933', 'system': 'http:...",<NA>
5,dad0eeb6-4063-5a5d-8aed-4a42319b767e,{'reference': 'Patient/00000027-c5e0-554f-8e85...,"{'coding': [{'code': '52033', 'system': 'http:...",<NA>
6,1c43fee5-7429-5835-ba27-178c582403c8,{'reference': 'Patient/00000027-c5e0-554f-8e85...,"{'coding': [{'code': '50802', 'system': 'http:...","{'code': 'mEq/L', 'unit': 'mEq/L', 'value': -2..."
7,32de9fdd-ea2f-5c13-a217-b0dfacd61210,{'reference': 'Patient/00000027-c5e0-554f-8e85...,"{'coding': [{'code': '52033', 'system': 'http:...",<NA>
8,82b1313b-5aa5-5a3f-bb67-39164ff42073,{'reference': 'Patient/00000027-c5e0-554f-8e85...,"{'coding': [{'code': '51484', 'system': 'http:...",<NA>
9,3fa3a43e-b4c4-5ca6-8937-f48c24878abe,{'reference': 'Patient/00000027-c5e0-554f-8e85...,"{'coding': [{'code': '51486', 'system': 'http:...",<NA>


### Test 3: Primary Key / Reference Integrity
Ensure that critical FHIR reference fields are populated and correctly formatted.

In [4]:
reference_check = conn.execute('''
    SELECT 
        COUNT(*) as total_events,
        SUM(CASE WHEN subject.reference IS NULL THEN 1 ELSE 0 END) as missing_patient_refs,
        SUM(CASE WHEN encounter.reference IS NULL THEN 1 ELSE 0 END) as missing_encounter_refs
    FROM bronze_chartevents
''').df()
display(reference_check)

,total_events,missing_patient_refs,missing_encounter_refs
0,313645032,0.0,0.0
